# Annie's Magic Numbers Code Challenge

**By José Agustín Moreno Larios**

## Introduction

The requirements for this report are to find out the following:

- Top 10 brands based on profits and margins
- Top 10 vendors based on profits and margins
- Which brands and vendors to drop due loses.

## Methodology

For this analysis, we will calculate the profits using the Cost Of Goods Sold (COGS) metric. In this way we can determine our best-selling brands and vendors.

From our SQL database exploration (refer to `notebooks/sql_columns_exploration.ipynb`), we do need to calculate two different COGS metrics: the full equation for the per-brand metrics, and a purchases-only COGS for the per-vendor one since the inventory tables do not contain vendor-specific information.

The full accounting COGS formula is:
$$COGS = Initial\ Inventory\ Value + Purchases + Freight\ Costs - Final\ Inventory\ Value$$

Then, the profit is:
$$Profit = Revenue - COGS$$

Thus, the margins are:
$$Margins = \frac{Revenue - COGS}{Revenue} \times 100\ [\%]$$

In [1]:
# Imports and setup
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

from mdutils.mdutils import MdUtils
import analysis
from config import Config

mdFile = MdUtils(file_name=str(Config.REPORT_PATH / 'report.md'),
                 title="Annie's Magic Numbers Code Challenge")
mdFile.author = "José Agustín Moreno"

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)

---

# Top 10 Brands

Freight costs are considered as a per-invoice basis, the data on the sales and purchases tables are shown as per-store.
We will aggregate this information into a per-brand basis so we can account for a proportional freight allocation for the COGS calculation.

From exploring the SQL tables, we identified that the brand refers to a single product type.

In [2]:
# Calculate
cogs_brand = analysis.calculate_cogs_per_brand()
summary_brand = analysis.calculate_brand_profits_margins(cogs_brand)

## Per profits

In [3]:
mdFile.new_header(level=1, title="Top 10 brands")
mdFile.new_header(level=2, title="Per profits")

top_profit = summary_brand[["Brand", "Description", "total_revenue", "cogs", "profit", "margin"]].nlargest(10, "profit")
mdFile.new_paragraph(top_profit.to_markdown(index=False))
top_profit

,Brand,Description,total_revenue,cogs,profit,margin
609,1233,Jack Daniels No 7 Black,5101919.51,3.751050e+06,1.350869e+06,26.477668
1802,3545,Ketel One Vodka,4223107.62,2.988901e+06,1.234207e+06,29.225088
2232,4261,Capt Morgan Spiced Rum,4475972.88,3.257282e+06,1.218691e+06,27.227401
3302,8068,Absolut 80 Proof,4538120.60,3.430854e+06,1.107267e+06,24.399233
1715,3405,Tito's Handmade Vodka,4819073.49,3.735514e+06,1.083559e+06,22.484807
3017,6570,Kendall Jackson Chard Vt RSV,2326007.78,1.457832e+06,8.681757e+05,37.324711
2014,3858,Grey Goose Vodka,3383912.40,2.533423e+06,8.504897e+05,25.133325
1769,3489,Tanqueray,2640491.19,1.885373e+06,7.551182e+05,28.597641
672,1376,Jim Beam,2435393.39,1.753061e+06,6.823319e+05,28.017319
1247,2663,Dewars White Label,2189368.78,1.509545e+06,6.798233e+05,31.051111


## Per margins

### Naive run — no purchases done in the period

In [4]:
mdFile.new_header(level=2, title="Per margins")
mdFile.new_header(level=3, title="Naive run - no purchases done in the period")

top_margin_naive = summary_brand[["Brand", "Description", "total_revenue", "cogs", "profit", "margin"]].nlargest(10, "margin")
mdFile.new_paragraph(top_margin_naive.to_markdown(index=False))
top_margin_naive

,Brand,Description,total_revenue,cogs,profit,margin
562,1099,Angel's Envy NH Blend Bourbn,6346.62,0.0,6346.62,100.0
596,1202,Hennessy VS Chain VAP,191.94,0.0,191.94,100.0
981,2166,The Macallan Double Cask 12,98245.68,0.0,98245.68,100.0
2170,4164,Hennessey 250 Collectors Edi,1199.98,0.0,1199.98,100.0
5431,18266,Gianni Gagliardo Barolo 08,95.98,0.0,95.98,100.0
6201,20275,Louis Jadot Les Drazeys 11,73.98,0.0,73.98,100.0
6352,20680,A Bichot Champs Martin,245.94,0.0,245.94,100.0
7161,22787,Tenuta La Fuga 09 Brun Montl,199.96,0.0,199.96,100.0
7277,23048,Dom Sigalas 11 Nychteri Assy,124.95,0.0,124.95,100.0
9882,33967,Prunotto Bric Touro Barbesco,3617.14,0.0,3617.14,100.0


> **Note:** These 100% margin brands had zero purchases during the period. Sales came entirely from existing inventory — this is a data artifact, not real pricing.

### Considering if brand was ordered in the period

In [5]:
mdFile.new_header(level=3, title="Considering if brand was ordered in the period")

top_margin_filtered = summary_brand[summary_brand['cogs'] > 0]
top_margin_filtered = top_margin_filtered[["Brand", "Description", "total_revenue", "cogs", "profit", "margin"]].nlargest(10, "margin")
mdFile.new_paragraph(top_margin_filtered.to_markdown(index=False))
top_margin_filtered

,Brand,Description,total_revenue,cogs,profit,margin
2619,5335,Beniotome Sesame Shochu,4768.41,22.323187,4746.086813,99.531853
10570,41231,Mad Dogs & Englishmen Jumil,279.80,6.563278,273.236722,97.654297
507,1020,B & B Dom VAP,1319.48,36.404585,1283.075415,97.240990
6353,20682,A Bichot Chablis Vaucopins,1623.44,59.883632,1563.556368,96.311312
10402,39461,Stags Leap SLV Cab Svgn,7034.33,416.531972,6617.798028,94.078584
695,1414,Bacardi 8 Gift Set,869.65,71.763870,797.886130,91.747960
599,1214,Apple Orchard Liqueur,2017.98,198.586910,1819.393090,90.159124
1221,2626,Crown Royal Apple,27.86,2.854473,25.005527,89.754224
5398,18130,Castello Di Ama Chianti,3107.16,324.435006,2782.724994,89.558471
10197,37421,Stags Leap Csk 23 Cab Svgn,11714.29,1399.918107,10314.371893,88.049484


## Losing brands

In [6]:
mdFile.new_header(level=2, title="Losing brands")

losing_brands = summary_brand[summary_brand["profit"] < 0].sort_values("profit")
mdFile.new_paragraph(losing_brands[["Brand",
                                    "Description",
                                    "total_revenue",
                                    "cogs", "profit", "margin"]]
                     .head(20)
                     .to_markdown(index=False)
                     )
losing_brands[["Brand", "Description", "total_revenue", "cogs", "profit", "margin"]].head(20)

,Brand,Description,total_revenue,cogs,profit,margin
8607,25588,High Valley Znfdl,55406.43,80577.583837,-25171.153837,-45.430023
9129,26710,Feudi Di San Gregorio Fiano,26032.05,46489.418682,-20457.368682,-78.585316
2259,4300,BenRiach Barrel 94,5039.72,20444.020930,-15404.300930,-305.657872
10946,44714,Buehler Znfdl Napa,119616.75,134791.909961,-15175.159961,-12.686484
9839,33331,Moletto Prosecco Della Marc,65065.64,78043.648520,-12978.008520,-19.946025
3790,10666,Clayhouse Adobe Cntrl Cst Wh,31188.64,43857.217538,-12668.577538,-40.619205
5994,19735,Beringer Quantum Red Napa Vl,31159.68,43524.975763,-12365.295763,-39.683642
644,1297,Jim Beam Black,18811.74,30093.787063,-11282.047063,-59.973437
666,1361,BenRiach 1994,12919.32,23335.697365,-10416.377365,-80.626359
8547,25508,Sbragia Zin Ginos Vyd- Dry C,32597.88,42098.736286,-9500.856286,-29.145626


## Brand Analysis — Key Results

### High Vodka and Whiskey sales drive most of the profit

Most of the profits are driven by high-volume sales, which are reflected on the first table.
From it, we can see that four Vodka brands and three Whiskey bands dominate the leaderboard.

### '100%' margins are inventory runoff

Brands that have 100% margin over this period are because no purchases were made.
Sales were made from existing stock, which means that these margins are an data artifact.

### High margins correspond to low-volume items

Brands with 90%+ margins correspond to tiny revenue scales. This is caused by existing inventory with minimal restocking. These products are not scalable profit drivers.

### Losing brands reflect a change in consumer taste

Most of the losing brands in the period are wines with high COGS, suggesting that Annie's may have overprovisioned the stock for the season.
We'd recommend to not order new stock on the losing brands until their COGS value gets lower in future months.

In [7]:
mdFile.new_header(level=2, title="Brand Analysis - Key Results")
mdFile.new_header(level=3, title="High Vodka and Whiskey sales drive most of the profit")
mdFile.new_paragraph(
    """
Most of the profits are driven by high-volume sales, which are reflected on the first table.
From it, we can see that four Vodka brands and three Whiskey bands dominate the leaderboard.
    """
)
mdFile.new_header(level=3, title="'100%' margins are inventory runoff")
mdFile.new_paragraph(
    """
Brands that have 100% margin over this period are because no purchases were made.
Sales were made from existing stock, which means that these margins are an data artifact.
    """
)
mdFile.new_header(level=3, title="High margins correspond to low-volume items")
mdFile.new_paragraph(
    """
Brands with 90%+ margins correspond to tiny revenue scales. This is caused by existing inventory with minimal restocking. These products are not scalable profit drivers.
    """
)
mdFile.new_header(level=3, title="Losing brands reflect a change in consumer taste")
mdFile.new_paragraph(
    """
Most of the losing brands in the period are wines with high COGS, suggesting that Annie's may have overprovisioned the stock for the season.
We'd recommend to not order new stock on the losing brands until their COGS value gets lower in future months.
    """
)

"\n# Top 10 brands\n\n## Per profits\n\n\n|   Brand | Description                  |   total_revenue |        cogs |           profit |   margin |\n|--------:|:-----------------------------|----------------:|------------:|-----------------:|---------:|\n|    1233 | Jack Daniels No 7 Black      |     5.10192e+06 | 3.75105e+06 |      1.35087e+06 |  26.4777 |\n|    3545 | Ketel One Vodka              |     4.22311e+06 | 2.9889e+06  |      1.23421e+06 |  29.2251 |\n|    4261 | Capt Morgan Spiced Rum       |     4.47597e+06 | 3.25728e+06 |      1.21869e+06 |  27.2274 |\n|    8068 | Absolut 80 Proof             |     4.53812e+06 | 3.43085e+06 |      1.10727e+06 |  24.3992 |\n|    3405 | Tito's Handmade Vodka        |     4.81907e+06 | 3.73551e+06 |      1.08356e+06 |  22.4848 |\n|    6570 | Kendall Jackson Chard Vt RSV |     2.32601e+06 | 1.45783e+06 | 868176           |  37.3247 |\n|    3858 | Grey Goose Vodka             |     3.38391e+06 | 2.53342e+06 | 850490           |  25.1333 |\n|   

---

# Top 10 Vendors

Since both beggining and end inventory tables do not have information regarding the vendor, we cannot use the full accounting formula.
Instead we use the purchase-based COGS:

$$COGS_{vendor} = Purchases_{vendor} + Freight_{vendor}$$

In [8]:
# Calculate
cogs_vendor = analysis.calculate_cogs_per_vendor()
summary_vendor = analysis.calculate_vendor_profits_margins(cogs_vendor)

## Per profits

In [9]:
mdFile.new_header(level=1, title="Top 10 vendors")
mdFile.new_header(level=2, title="Per profits")

top_profit_vendor = summary_vendor[["VendorNumber", "VendorName", "total_revenue", "cogs", "profit", "margin"]].nlargest(10, "profit")
mdFile.new_paragraph(top_profit_vendor.to_markdown(index=False))
top_profit_vendor

,VendorNumber,VendorName,total_revenue,cogs,profit,margin
40,3960,DIAGEO NORTH AMERICA INC,68742416.99,51216828.92,17525588.07,25.494576
42,4425,MARTIGNETTI COMPANIES,41047306.30,27966193.83,13081112.47,31.868382
16,1392,CONSTELLATION BRANDS INC,24469172.93,15653446.89,8815726.04,36.027887
99,17035,PERNOD RICARD USA,32281247.95,24247871.78,8033376.17,24.885581
95,12546,JIM BEAM BRANDS COMPANY,31906320.54,24327032.02,7579288.52,23.754818
6,480,BACARDI USA INC,25014556.89,17713664.99,7300891.90,29.186573
34,3252,E & J GALLO WINERY,18556085.61,12351575.00,6204510.61,33.436527
12,1128,BROWN-FORMAN CORP,18478557.47,13598034.76,4880522.71,26.411817
79,9165,ULTRA BEVERAGE COMPANY LLP,17822938.45,13278668.63,4544269.82,25.496749
82,9552,M S WALKER INC,15465247.75,10991369.12,4473878.63,28.928593


## Per margins

### Naive run — no purchases done in the period

In [10]:
mdFile.new_header(level=2, title="Per margins")
mdFile.new_header(level=3, title="Naive run - no purchases done in the period")

top_margin_vendor_naive = summary_vendor[["VendorNumber", "VendorName", "total_revenue", "cogs", "profit", "margin"]].nlargest(10, "margin")
mdFile.new_paragraph(top_margin_vendor_naive.to_markdown(index=False))
top_margin_vendor_naive

,VendorNumber,VendorName,total_revenue,cogs,profit,margin
10,1002,BERNIKO LLC,16.99,0.00,16.99,100.000000
85,9710,WHYTE & MACKAY,31.98,0.00,31.98,100.000000
111,90034,EXCLUSIVE WINES & SPIRITS,55.98,0.00,55.98,100.000000
126,201359,FLAVOR ESSENCE INC,1474.41,17.09,1457.32,98.840892
107,90026,SILVER MOUNTAIN CIDERS,381.48,77.54,303.94,79.673902
17,1439,CAPSTONE INTERNATIONAL,246.87,54.91,191.96,77.757524
24,1703,ALISA CARR BEVERAGES,118167.38,35123.68,83043.70,70.276332
72,8663,STAR INDUSTRIES INC.,7914.72,2464.73,5449.99,68.858911
112,90037,THE PIERPONT GROUP LLC,17937.21,5741.17,12196.04,67.992960
66,7749,R.P.IMPORTS INC,54266.62,18868.37,35398.25,65.230247


### Considering if we ordered from a given vendor during the period

In [11]:
mdFile.new_header(level=3, title="Considering if we ordered from a given vendor during the period")

top_margin_vendor_filtered = summary_vendor[summary_vendor['cogs'] > 0]
top_margin_vendor_filtered = top_margin_vendor_filtered[["VendorNumber", "VendorName", "total_revenue", "cogs", "profit", "margin"]].nlargest(10, "margin")
mdFile.new_paragraph(top_margin_vendor_filtered.to_markdown(index=False))
top_margin_vendor_filtered

,VendorNumber,VendorName,total_revenue,cogs,profit,margin
126,201359,FLAVOR ESSENCE INC,1474.41,17.09,1457.32,98.840892
107,90026,SILVER MOUNTAIN CIDERS,381.48,77.54,303.94,79.673902
17,1439,CAPSTONE INTERNATIONAL,246.87,54.91,191.96,77.757524
24,1703,ALISA CARR BEVERAGES,118167.38,35123.68,83043.70,70.276332
72,8663,STAR INDUSTRIES INC.,7914.72,2464.73,5449.99,68.858911
112,90037,THE PIERPONT GROUP LLC,17937.21,5741.17,12196.04,67.992960
66,7749,R.P.IMPORTS INC,54266.62,18868.37,35398.25,65.230247
110,90033,FANTASY FINE WINES CORP,327.59,129.25,198.34,60.545194
87,9751,VINEDREA WINES LLC,11385.60,4682.13,6703.47,58.876739
27,2396,BLACK PRINCE DISTILLERY INC,11818.85,6002.92,5815.93,49.208933


## Losing Vendors

In [12]:
mdFile.new_header(level=2, title="Losing Vendors")

losing_vendors = summary_vendor[summary_vendor["profit"] < 0].sort_values("profit")
mdFile.new_paragraph(losing_vendors[["VendorNumber",
                                    "VendorName",
                                    "total_revenue",
                                    "cogs", "profit", "margin"]]
                     .head(20)
                     .to_markdown(index=False)
                     )
losing_vendors[["VendorNumber", "VendorName", "total_revenue", "cogs", "profit", "margin"]].head(20)

,VendorNumber,VendorName,total_revenue,cogs,profit,margin
1,60,ADAMBA IMPORTS INTL INC,67576.22,77137.77,-9561.55,-14.149282
121,90059,BLACK COVE BEVERAGES,6256.87,14539.90,-8283.03,-132.382965
0,2,"IRA GOLDMAN AND WILLIAMS, LLP",1265.58,5657.96,-4392.38,-347.064587
39,3951,HIGHLAND WINE MERCHANTS LLC,1533.68,5529.75,-3996.07,-260.554353
55,6280,UNCORKED,1124.38,2981.57,-1857.19,-165.174585
35,3551,GILMANTON WINERY & VINEYARD,3837.60,5419.67,-1582.07,-41.225506
48,5083,LOYAL DOG WINERY,1111.26,2331.48,-1220.22,-109.805086
125,173357,TAMWORTH DISTILLING,40021.12,41238.94,-1217.82,-3.042943
4,287,APPOLO VINEYARDS LLC,1616.92,2411.98,-795.06,-49.171264
123,99166,STARK BREWING COMPANY,25371.54,26091.13,-719.59,-2.836209


## Vendor Analysis — Key Results

### Diageo and Martignetti dominate profits

Diageo North America generates 17.5 million in profit; Martignetti Companies, 13.1 million.
These two companies represent the majority of the top 10 earners.

### Losing vendors are small contributors

All losing vendors are small producers with revenue under 70k. The only company worth reviewing are Adamba Imports (67.6k revenue, -9.6k loss).

### No major vendor relationships need termination

All 10 profit-driving vendors are healthy. Losing vendor losses can be either ignored or fixed through pricing.

In [13]:
mdFile.new_header(level=2, title="Vendor Analysis - Key Results")
mdFile.new_header(level=3, title="Diageo and Martignetti dominate profits")

profit_share = (top_profit_vendor["profit"].iloc[0] + top_profit_vendor["profit"].iloc[1]) / top_profit_vendor["profit"].sum() * 100
mdFile.new_paragraph(f"""
Diageo North America generates 17.5 million in profit; Martignetti Companies, 13.1 million.
These two companies represent the {profit_share}% of the top 10 earners.
""")

mdFile.new_header(level=3, title="Losing vendors are small contributors")
mdFile.new_paragraph(f"""
All losing vendors are small producers with revenue under 70k. The only company worth reviewing are Adamba Imports (67.6k revenue, -9.6k loss).
""")

mdFile.new_header(level=3, title="No major vendor relationships need termination")
mdFile.new_paragraph(f"""
All 10 profit-driving vendors are healthy. Losing vendor losses can be either ignored or fixed through pricing.
""")

"\n# Top 10 brands\n\n## Per profits\n\n\n|   Brand | Description                  |   total_revenue |        cogs |           profit |   margin |\n|--------:|:-----------------------------|----------------:|------------:|-----------------:|---------:|\n|    1233 | Jack Daniels No 7 Black      |     5.10192e+06 | 3.75105e+06 |      1.35087e+06 |  26.4777 |\n|    3545 | Ketel One Vodka              |     4.22311e+06 | 2.9889e+06  |      1.23421e+06 |  29.2251 |\n|    4261 | Capt Morgan Spiced Rum       |     4.47597e+06 | 3.25728e+06 |      1.21869e+06 |  27.2274 |\n|    8068 | Absolut 80 Proof             |     4.53812e+06 | 3.43085e+06 |      1.10727e+06 |  24.3992 |\n|    3405 | Tito's Handmade Vodka        |     4.81907e+06 | 3.73551e+06 |      1.08356e+06 |  22.4848 |\n|    6570 | Kendall Jackson Chard Vt RSV |     2.32601e+06 | 1.45783e+06 | 868176           |  37.3247 |\n|    3858 | Grey Goose Vodka             |     3.38391e+06 | 2.53342e+06 | 850490           |  25.1333 |\n|   

In [14]:
# Generate the file
mdFile.create_md_file()
print(f"Report saved to {Config.REPORT_PATH / 'report.md'}")

Report saved to /home/jose/Documents/Búsqueda Laboral/BaseLabs/base-labs-liquour-analysis/reports/report.md
